In [ ]:
!nvidia-smi
!git clone https://github.com/Dobrosav/json2openapi-llm.git
%cd /content/json2openapi-llm
!pip install -q -r requirements.txt "transformers>=4.46,<5" accelerate datasets scikit-learn

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_ID = "Salesforce/codet5-small"
PREFIX = "Generate OpenAPI 3.0 YAML schema: "

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID)

dataset = load_dataset("json", data_files={
    "train": "data/train.jsonl",
    "validation": "data/val.jsonl",
    "test": "data/test.jsonl",
})

def preprocess(batch):
    inputs = [PREFIX + x for x in batch["input_json"]]
    model_inputs = tokenizer(inputs, max_length=384, truncation=True)
    labels = tokenizer(text_target=batch["target_yaml"], max_length=384, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized = dataset.map(preprocess, batched=True, remove_columns=["input_json", "target_yaml"])
print(tokenized)

In [ ]:
import os, json
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir="./codet5-openapi",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    learning_rate=3e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    fp16=False,
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=data_collator,
)

trainer.train()

trainer.save_model("./codet5-openapi-best")
tokenizer.save_pretrained("./codet5-openapi-best")

os.makedirs("results", exist_ok=True)
with open("results/codet5_training_log.json", "w", encoding="utf-8") as f:
    json.dump(trainer.state.log_history, f, indent=2)

In [ ]:
import time
import torch
from evaluate_ollama import evaluate_prediction

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

with open("data/test.jsonl", "r", encoding="utf-8") as f:
    test_data = [json.loads(line) for line in f]

records = []
t0_all = time.time()

for i, row in enumerate(test_data, 1):
    t0 = time.time()
    inp = tokenizer(
        PREFIX + row["input_json"],
        return_tensors="pt",
        truncation=True,
        max_length=384,
    ).to(device)

    with torch.no_grad():
        out_ids = model.generate(**inp, max_new_tokens=384, num_beams=2)

    pred_yaml = tokenizer.decode(out_ids[0], skip_special_tokens=True).strip()
    elapsed = round(time.time() - t0, 3)

    metrics = evaluate_prediction(row["input_json"], row["target_yaml"], pred_yaml)
    records.append({
        "index": i,
        "latency_s": elapsed,
        "pred_yaml": pred_yaml,
        **metrics,
    })

    if i % 20 == 0 or i == len(test_data):
        print(
            f"  [{i}/{len(test_data)}] "
            f"OAS_valid={sum(r['oas_valid'] for r in records)}/{i}, "
            f"JSON_valid={sum(r['json_valid'] for r in records)}/{i}, "
            f"Exact={sum(r['exact_match'] for r in records)}/{i}"
        )

n = len(records)
stats = {
    "count": n,
    "yaml_parse_rate": round(100 * sum(r["yaml_valid"] for r in records) / n, 2),
    "oas_schema_valid_rate": round(100 * sum(r["oas_valid"] for r in records) / n, 2),
    "json_validation_accuracy": round(100 * sum(r["json_valid"] for r in records) / n, 2),
    "exact_match_rate": round(100 * sum(r["exact_match"] for r in records) / n, 2),
    "avg_latency_s": round(sum(r["latency_s"] for r in records) / n, 3),
    "total_time_s": round(time.time() - t0_all, 2),
}

with open("results/transformer_codet5.json", "w", encoding="utf-8") as out:
    json.dump({"stats": stats, "records": records}, out, indent=2, ensure_ascii=False)

print("\n=== FINAL SUMMARY (CodeT5-small) ===")
print(json.dumps(stats, indent=2))

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!cp results/transformer_codet5.json /content/drive/MyDrive/json2openapi-llm/results/
!cp results/codet5_training_log.json /content/drive/MyDrive/json2openapi-llm/results/
print("Sačuvano na Google Drive!")

In [ ]:
import json, yaml, torch
from prepare_dataset import infer_openapi_schema
from evaluate_ollama import evaluate_prediction

ood_samples = [
    {"order_id": "ORD-991", "total_eur": 149.99, "paid": True, "coupon": None, "items": [{"sku": "A1", "qty": 2}]},
    {"patient": {"jmbg_hash": "abc123", "age": 42, "smoker": False}, "allergies": [], "visits": 3},
    {"coords": {"lat": 44.8125, "lon": 20.4612}, "city": "Belgrade", "alerts": ["wind", "rain"]},
    {"tx_id": 884120, "amount": 2500.5, "currency": "RSD", "approved": True, "meta": {"branch": "Zvezdara", "retry": 0}},
    {"sensor": "DHT22", "readings": [{"temp_c": 23.4, "ok": True}], "notes": None, "errors": []},
]

ood_results = []
for idx, obj in enumerate(ood_samples, 1):
    inp_str = json.dumps(obj, ensure_ascii=False)
    target_yaml = yaml.safe_dump(infer_openapi_schema(obj), sort_keys=False).strip()

    inp = tokenizer(PREFIX + inp_str, return_tensors="pt", truncation=True, max_length=384).to(device)
    with torch.no_grad():
        out_ids = model.generate(**inp, max_new_tokens=384, num_beams=2)
    pred_yaml = tokenizer.decode(out_ids[0], skip_special_tokens=True).strip()

    m = evaluate_prediction(inp_str, target_yaml, pred_yaml)
    ood_results.append({"index": idx, "input_json": inp_str, "pred_yaml": pred_yaml, **m})
    print(f"[OOD {idx}/5] OAS={m['oas_valid']} | JSON={m['json_valid']} | Exact={m['exact_match']} | err={m['error']}")

with open("results/codet5_ood_test.json", "w", encoding="utf-8") as f:
    json.dump(ood_results, f, indent=2, ensure_ascii=False)

!cp results/codet5_ood_test.json /content/drive/MyDrive/json2openapi-llm/results/